# 03 — Classification modelling: cutoffs, class imbalance and intervention objective

## Objective
Compare 7-, 14- and 30-day snapshots. The primary objective is **maximize recall subject to a minimum precision floor**, not maximize F1. Precision floors of 0.40, 0.50 and 0.60 are reported as sensitivity analyses; choose the operational floor based on outreach capacity and cost.

## Evaluation design
- One row per student; the same stratified student split is reused across cutoffs.
- 60% train, 20% validation, 20% final test.
- The validation set selects the threshold. The test set is not used for threshold selection.
- Compare no class correction, class weighting, and random oversampling applied to training rows only.
- Report PR-AUC, ROC-AUC, precision, recall, F1 (secondary), confusion matrix, fraction flagged, and whether the precision floor was achieved.

## Important target limitation
`outcome.csv` has a student-level label but no dropout timestamp. These metrics are against the supplied label; they do not prove the label occurred after the snapshot cutoff.


In [1]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score, roc_auc_score, confusion_matrix
from IPython.display import display

ROOT_CANDIDATES = [Path(".."), Path("."), Path("/mnt/data/student_success_assistant_v2")]
ROOT = next((p.resolve() for p in ROOT_CANDIDATES if (p / "data" / "curated").exists()), Path(".").resolve())
DATA = ROOT / "data" / "curated"
OUT = ROOT / "outputs" / "metrics"
OUT.mkdir(parents=True, exist_ok=True)
PRECISION_FLOORS = [0.40, 0.50, 0.60]
RANDOM_STATE = 42


In [2]:
files = {d: pd.read_csv(DATA / f"classification_snapshot_{d}d.csv") for d in (7, 14, 30)}
for days, df in files.items():
    assert len(df) == 500 and df.student_id.is_unique
    assert df.target_dropout.isin([0,1]).all()
    print(f"{days}-day snapshot: {len(df)} students; positives={int(df.target_dropout.sum())}; prevalence={df.target_dropout.mean():.3f}")


7-day snapshot: 500 students; positives=125; prevalence=0.250
14-day snapshot: 500 students; positives=125; prevalence=0.250
30-day snapshot: 500 students; positives=125; prevalence=0.250


## 1. Student-level split, shared across cutoffs

The same student IDs appear in all three snapshots. We split IDs once and reuse the partitions, so differences are attributable to cutoff features rather than different students. This is a random student split (interpolation), not a chronological future-cohort evaluation.

In [3]:
reference = files[14].set_index("student_id")
ids = reference.index.to_numpy()
y_ids = reference["target_dropout"].astype(int)
train_ids, temp_ids = train_test_split(ids, test_size=0.40, random_state=RANDOM_STATE, stratify=y_ids.loc[ids])
temp_y = y_ids.loc[temp_ids]
val_ids, test_ids = train_test_split(temp_ids, test_size=0.50, random_state=RANDOM_STATE, stratify=temp_y)
split_ids = {"train":set(train_ids), "validation":set(val_ids), "test":set(test_ids)}
assert all(not (split_ids[a] & split_ids[b]) for a,b in [("train","validation"),("train","test"),("validation","test")])
split_report = pd.DataFrame([{"split":k,"students":len(v),"positives":int(y_ids.loc[list(v)].sum()),"prevalence":y_ids.loc[list(v)].mean()} for k,v in split_ids.items()])
display(split_report)
split_report.to_csv(OUT / "classification_split_report.csv", index=False)


,split,students,positives,prevalence
0,train,300,75,0.25
1,validation,100,25,0.25
2,test,100,25,0.25


## 2. Pipelines and training-only imbalance strategies

`none` uses the natural training distribution. `class_weight` changes the loss weighting. `random_oversampling` samples positive training rows with replacement until classes are balanced. Oversampling occurs **after the train/validation/test split and only on the training partition**. Validation and test retain the natural class distribution.

SMOTE is intentionally deferred: these are mixed categorical and numeric features, and synthetic interpolations can create implausible records. First establish whether simpler methods improve recall at the precision floor.

In [4]:
EXCLUDED = {"student_id", "target_dropout", "snapshot_days", "snapshot_cutoff"}

def build_pipeline(df, model_name, imbalance):
    features = [c for c in df.columns if c not in EXCLUDED]
    cat = [c for c in features if df[c].dtype == "object" or str(df[c].dtype).startswith(("category","string"))]
    num = [c for c in features if c not in cat]
    numeric = Pipeline([("impute",SimpleImputer(strategy="median", add_indicator=True)),("scale",StandardScaler())])
    categorical = Pipeline([("impute",SimpleImputer(strategy="most_frequent")),("onehot",OneHotEncoder(handle_unknown="ignore"))])
    prep = ColumnTransformer([("num",numeric,num),("cat",categorical,cat)])
    if model_name == "Logistic Regression":
        cw = "balanced" if imbalance == "class_weight" else None
        clf = LogisticRegression(max_iter=3000, class_weight=cw, random_state=RANDOM_STATE)
    else:
        cw = "balanced_subsample" if imbalance == "class_weight" else None
        clf = RandomForestClassifier(n_estimators=300, min_samples_leaf=4, class_weight=cw, random_state=RANDOM_STATE, n_jobs=-1)
    return Pipeline([("prep",prep),("model",clf)]), features

def oversample_training(X, y, seed=RANDOM_STATE):
    frame = X.copy()
    frame["__target__"] = np.asarray(y)
    counts = frame["__target__"].value_counts()
    majority = counts.max()
    pieces = [frame]
    rng = np.random.default_rng(seed)
    for cls, count in counts.items():
        if count < majority:
            part = frame[frame["__target__"] == cls]
            extra = part.iloc[rng.integers(0, len(part), size=majority-count)]
            pieces.append(extra)
    res = pd.concat(pieces, ignore_index=True).sample(frac=1, random_state=seed)
    return res.drop(columns="__target__"), res["__target__"].astype(int)

def get_scores(y, p, threshold):
    pred = p >= threshold
    cm = confusion_matrix(y, pred, labels=[0,1])
    try: pr = average_precision_score(y,p)
    except ValueError: pr = np.nan
    try: roc = roc_auc_score(y,p)
    except ValueError: roc = np.nan
    precision = precision_score(y,pred,zero_division=0)
    return {"precision":precision,"recall":recall_score(y,pred,zero_division=0),
            "f1_secondary":f1_score(y,pred,zero_division=0),"PR_AUC":pr,"ROC_AUC":roc,
            "flagged_rate":float(pred.mean()),"TN":int(cm[0,0]),"FP":int(cm[0,1]),
            "FN":int(cm[1,0]),"TP":int(cm[1,1])}

def choose_recall_threshold(y, p, precision_floor):
    thresholds = np.unique(np.r_[0.01, np.arange(0.025,0.976,0.025), 0.99])
    candidates = []
    for t in thresholds:
        pred = p >= t
        prec = precision_score(y,pred,zero_division=0)
        rec = recall_score(y,pred,zero_division=0)
        if pred.sum() and prec >= precision_floor:
            candidates.append((rec,prec,t))
    if not candidates:
        return np.nan, False
    rec, prec, threshold = max(candidates, key=lambda z:(z[0],z[1],z[2]))
    return float(threshold), True


## 3. Train, select threshold on validation, evaluate once on test

A threshold is selected independently for each precision floor using validation predictions only. If no validation threshold meets the floor, that model/setting is marked infeasible for that floor; we do not quietly substitute a threshold that violates the requirement.

In [5]:
result_rows = []
for days, raw in files.items():
    df = raw.set_index("student_id").loc[ids].reset_index()
    parts = {k:df[df.student_id.isin(v)].copy() for k,v in split_ids.items()}
    ytr = parts["train"].target_dropout.astype(int)
    yv = parts["validation"].target_dropout.astype(int)
    yt = parts["test"].target_dropout.astype(int)
    for model_name in ["Logistic Regression","Random Forest"]:
        for imbalance in ["none","class_weight","random_oversampling"]:
            model, features = build_pipeline(df,model_name,imbalance)
            Xtr = parts["train"][features]
            if imbalance == "random_oversampling":
                Xfit, yfit = oversample_training(Xtr,ytr)
            else:
                Xfit, yfit = Xtr,ytr
            model.fit(Xfit,yfit)
            pv = model.predict_proba(parts["validation"][features])[:,1]
            pt = model.predict_proba(parts["test"][features])[:,1]
            for floor in PRECISION_FLOORS:
                threshold, feasible = choose_recall_threshold(yv,pv,floor)
                if feasible:
                    met = get_scores(yt,pt,threshold)
                else:
                    met = {k:np.nan for k in ["precision","recall","f1_secondary","flagged_rate","TN","FP","FN","TP"]}
                    met.update({"PR_AUC":average_precision_score(yt,pt),"ROC_AUC":roc_auc_score(yt,pt)})
                result_rows.append({"snapshot_days":days,"model":model_name,"imbalance_strategy":imbalance,
                    "precision_floor":floor,"validation_feasible":feasible,"threshold":threshold,
                    "validation_recall_at_threshold":recall_score(yv,pv>=threshold,zero_division=0) if feasible else np.nan,
                    **{f"test_{k}":v for k,v in met.items()}})
results = pd.DataFrame(result_rows)
display(results.sort_values(["precision_floor","snapshot_days","test_recall"],ascending=[True,True,False]).round(3))
results.to_csv(OUT / "classification_imbalance_precision_floor_comparison.csv",index=False)


,snapshot_days,model,imbalance_strategy,precision_floor,validation_feasible,threshold,validation_recall_at_threshold,test_precision,test_recall,test_f1_secondary,test_PR_AUC,test_ROC_AUC,test_flagged_rate,test_TN,test_FP,test_FN,test_TP
3,7,Logistic Regression,class_weight,0.4,True,0.325,0.80,0.357,0.80,0.494,0.534,0.719,0.56,39.0,36.0,5.0,20.0
0,7,Logistic Regression,none,0.4,True,0.150,0.76,0.339,0.76,0.469,0.535,0.715,0.56,38.0,37.0,6.0,19.0
9,7,Random Forest,none,0.4,True,0.225,0.68,0.347,0.68,0.459,0.463,0.669,0.49,43.0,32.0,8.0,17.0
12,7,Random Forest,class_weight,0.4,True,0.300,0.72,0.309,0.68,0.425,0.484,0.660,0.55,37.0,38.0,8.0,17.0
15,7,Random Forest,random_oversampling,0.4,True,0.375,0.60,0.370,0.68,0.479,0.468,0.660,0.46,46.0,29.0,8.0,17.0
6,7,Logistic Regression,random_oversampling,0.4,True,0.450,0.64,0.395,0.60,0.476,0.518,0.687,0.38,52.0,23.0,10.0,15.0
30,14,Random Forest,class_weight,0.4,True,0.300,0.84,0.375,0.72,0.493,0.540,0.700,0.48,45.0,30.0,7.0,18.0
18,14,Logistic Regression,none,0.4,True,0.125,0.88,0.333,0.68,0.447,0.621,0.733,0.51,41.0,34.0,8.0,17.0
21,14,Logistic Regression,class_weight,0.4,True,0.275,0.88,0.309,0.68,0.425,0.620,0.728,0.55,37.0,38.0,8.0,17.0
24,14,Logistic Regression,random_oversampling,0.4,True,0.275,0.80,0.327,0.68,0.442,0.610,0.723,0.52,40.0,35.0,8.0,17.0


## 4. Operational reference policies

The “contact everyone” policy has 100% recall, precision equal to the positive prevalence, and a 100% flagged rate. It is not a learned model; it is an intervention-capacity reference. The conventional majority classifier predicts the negative class for everyone at its default threshold and has zero recall for the positive class.

In [6]:
test_df = files[14][files[14].student_id.isin(split_ids["test"])]
test_prevalence = test_df.target_dropout.mean()
policy_refs = pd.DataFrame([
    {"policy":"contact_everyone","precision":test_prevalence,"recall":1.0,"flagged_rate":1.0,
     "note":"Operational upper bound on recall; contacts all students"},
    {"policy":"majority_negative","precision":np.nan,"recall":0.0,"flagged_rate":0.0,
     "note":"Conventional majority-class prediction at default threshold"}
])
display(policy_refs)
policy_refs.to_csv(OUT / "classification_policy_baselines.csv",index=False)


,policy,precision,recall,flagged_rate,note
0,contact_everyone,0.25,1.0,1.0,Operational upper bound on recall; contacts al...
1,majority_negative,NaN,0.0,0.0,Conventional majority-class prediction at defa...


## 5. Interpretation checklist

- Primary decision rule: maximize recall subject to precision ≥ the chosen floor.
- Report whether a floor was feasible on validation. A threshold that meets a floor on a small validation set may not meet it on the test set.
- Compare the same cutoff/model/imbalance combination across floors; test precision can fall below the floor because the constraint was selected on validation, not test.
- Test results are from one stratified split of 100 students (about 25 positives), so they are noisy. Use repeated student-level cross-validation on development data before final model selection; retain the final test set for one final report.
- This is random student interpolation. A separate chronological cohort evaluation is needed to assess transfer to later cohorts.
- The target has no timestamp, so these results do not validate that dropout occurs after the snapshot.
- Do not choose oversampling simply because it raises recall; compare precision, flagged rate, PR-AUC and stability across splits.
